# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = "/content/drive/MyDrive/juuzou-wake"
!mkdir -p {SAVE}
!ls {SAVE}

Mounted at /content/drive
clips.tar	  negative_features_test.npy   positive_features_train.npy
juuzou.onnx	  negative_features_train.npy
juuzou.onnx.data  positive_features_test.npy


In [4]:
!mkdir -p {SAVE}/v1-juuzou && mv {SAVE}/clips.tar {SAVE}/*.npy {SAVE}/juuzou.onnx* {SAVE}/v1-juuzou/ 2>/dev/null
!ls {SAVE}

v1-juuzou


In [5]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 4.82 MiB/s, done.
Resolving deltas: 100% (93/93), done.
--2026-09-02 17:38:28--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-02T18%3A16%3A47Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-99

In [6]:
!pip install -q onnxruntime onnxscript piper-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 10.9 MB/s eta 0:00:00


In [7]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [8]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [03:43,  1.21it/s]


In [10]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


--2026-09-02 17:44:54--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.40, 13.35.202.121, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-09-02 17:44:54 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]


 99%|█████████▉| 119/120 [00:43<00:00,  2.75it/s]


In [11]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-09-02 17:46:18--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1788374778&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD9yZXNwb25zZS1jb250ZW50LWRpc3B

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [14]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [15]:
!pwd && ls my_model.yaml

/content
ls: cannot access 'my_model.yaml': No such file or directory


In [16]:
# Modify values in the config and save a new version

import yaml

try:
    config
except NameError:
    config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)

config["target_phrase"] = ["suzuya", "soozooya", "sue zoo ya", "suzooya", "soo zoo ya"]
config["model_name"] = "suzuya"
config["n_samples"] = 10000
config["n_samples_val"] = 2000
config["steps"] = 25000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("записано:", config["target_phrase"], config["model_name"])
!ls -l my_model.yaml

записано: ['suzuya', 'soozooya', 'sue zoo ya', 'suzooya', 'soo zoo ya'] suzuya
-rw-r--r-- 1 root root 802 Sep  2 18:00 my_model.yaml


In [17]:
import re, pathlib, glob

# generate_samples.py исчез из piper-sample-generator — откат на родительский коммит
!git -C /content/piper-sample-generator checkout -q c9d824c

tp = pathlib.Path("/content/openwakeword/openwakeword/train.py")
s = tp.read_text()

# generate_samples() требует путь к голосовой модели piper (вызовов четыре)
VOICE = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
if VOICE not in s:
    s, n = re.subn(r"generate_samples\(\s*", f'generate_samples(model="{VOICE}", ', s)
    print("правок generate_samples:", n, "(ожидается 4)")

# torch.load не грузит чекпоинт DeepPhonemizer
if "weights_only=False" not in s:
    s = s.replace("import openwakeword",
                  "import functools, torch\ntorch.load = functools.partial(torch.load, weights_only=False)\nimport openwakeword", 1)

# torchaudio.info / torchaudio.load выпилены — обёртки над soundfile
SHIM = '''
# _sf_shim
import torchaudio as _ta, soundfile as _sf, torch as _torch
class _SfInfo:
    def __init__(self, p):
        i = _sf.info(p)
        self.sample_rate = i.samplerate
        self.num_frames = i.frames
        self.num_channels = i.channels
        self.bits_per_sample = 16
        self.encoding = "PCM_S"
def _sf_info(p, *a, **k):
    return _SfInfo(p)
def _sf_load(p, frame_offset=0, num_frames=-1, *a, **k):
    d, sr = _sf.read(p, start=frame_offset,
                     frames=(-1 if num_frames in (-1, None) else num_frames),
                     dtype="float32", always_2d=True)
    return _torch.from_numpy(d.T.copy()), sr
_ta.info = _sf_info
_ta.load = _sf_load
'''
if "_sf_shim" not in s:
    s = s.replace("import openwakeword", SHIM + "\nimport openwakeword", 1)

tp.write_text(s)

# torchaudio.set_audio_backend больше не существует
for f in glob.glob("/usr/local/lib/python3*/dist-packages/torch_audiomentations/utils/io.py"):
    io = pathlib.Path(f)
    t = io.read_text()
    if "set_audio_backend" in t and "# patched" not in t:
        io.write_text(re.sub(r"^(\s*)(torchaudio\.set_audio_backend.*)$",
                             r"\1pass  # patched: \2", t, flags=re.M))
        print("io.py пропатчен:", f)

print("generate_samples.py:", pathlib.Path("/content/piper-sample-generator/generate_samples.py").exists())

правок generate_samples: 4 (ожидается 4)
io.py пропатчен: /usr/local/lib/python3.13/dist-packages/torch_audiomentations/utils/io.py
generate_samples.py: True


In [18]:
!pip install -q onnxruntime onnxscript piper-tts

In [19]:
import importlib
mods = ["onnxruntime", "onnxscript", "piper", "torch", "torchaudio", "torchinfo",
        "torchmetrics", "speechbrain", "audiomentations", "torch_audiomentations",
        "acoustics", "pronouncing", "mutagen", "dp", "datasets", "tqdm",
        "scipy", "soundfile", "yaml"]
missing = []
for m in mods:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append((m, type(e).__name__))
print("не хватает:", missing or "ничего")
!nvidia-smi -L

не хватает: ничего
GPU 0: Tesla T4 (UUID: GPU-6610fe9b-25f4-7106-70fb-376b1f8869bb)


In [20]:
import functools, glob, inspect, os, shutil, sys, torch
torch.load = functools.partial(torch.load, weights_only=False)
sys.path.insert(0, "/content/piper-sample-generator")
from generate_samples import generate_samples
from IPython.display import Audio, display

print(inspect.signature(generate_samples))

VOICE = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
CANDIDATES = ["suzuya", "soozooya", "sue zoo ya", "suzooya", "soo zoo ya"]

for phrase in CANDIDATES:
    out = "/content/probe/" + phrase.replace(" ", "_")
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out, exist_ok=True)
    generate_samples([phrase], model=VOICE, max_samples=3, batch_size=3, output_dir=out)
    print("=" * 30, repr(phrase))
    for f in sorted(glob.glob(out + "/*.wav")):
        display(Audio(f))

(text: Union[List[str], str], output_dir: Union[str, pathlib._local.Path], model: Union[str, pathlib._local.Path], max_samples: Optional[int] = None, file_names: Optional[collections.abc.Iterable[str]] = None, batch_size: int = 1, slerp_weights: Tuple[float, ...] = (0.5,), length_scales: Tuple[float, ...] = (0.75, 1, 1.25), noise_scales: Tuple[float, ...] = (0.667,), noise_scale_ws: Tuple[float, ...] = (0.8,), max_speakers: Optional[int] = None, verbose: bool = False, phoneme_input: bool = False, **kwargs) -> None
============================== 'suzuya'


============================== 'soozooya'


============================== 'sue zoo ya'


============================== 'suzooya'


============================== 'soo zoo ya'


In [2]:
!ls /content && ls /content/my_custom_model 2>/dev/null

sample_data


In [13]:
!cat my_model.yaml | head -20

augmentation_batch_size: 16
augmentation_rounds: 1
background_paths:
- ./audioset_16k
- ./fma
background_paths_duplication_rate:
- 1
batch_n_per_class:
  ACAV100M_sample: 1024
  adversarial_negative: 50
  positive: 50
custom_negative_phrases: []
false_positive_validation_data_path: validation_set_features.npy
feature_data_files:
  ACAV100M_sample: openwakeword_features_ACAV100M_2000_hrs_16bit.npy
layer_size: 32
max_negative_weight: 1500
model_name: juuzou
model_type: dnn
n_samples: 10000


In [8]:
import re, pathlib, glob

# generate_samples.py исчез из piper-sample-generator — откат на родительский коммит
!git -C /content/piper-sample-generator checkout -q c9d824c

tp = pathlib.Path("/content/openwakeword/openwakeword/train.py")
s = tp.read_text()

# generate_samples() требует путь к голосовой модели piper
VOICE = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
if VOICE not in s:
    s, n = re.subn(r"generate_samples\(\s*", f'generate_samples(model="{VOICE}", ', s)
    print("правок generate_samples:", n, "(ожидается 4)")

# torch.load не грузит чекпоинт DeepPhonemizer
if "weights_only=False" not in s:
    s = s.replace("import openwakeword",
                  "import functools, torch\ntorch.load = functools.partial(torch.load, weights_only=False)\nimport openwakeword", 1)

# torchaudio.info / torchaudio.load выпилены — обёртки над soundfile
SHIM = '''
# _sf_shim
import torchaudio as _ta, soundfile as _sf, torch as _torch
class _SfInfo:
    def __init__(self, p):
        i = _sf.info(p)
        self.sample_rate = i.samplerate
        self.num_frames = i.frames
        self.num_channels = i.channels
        self.bits_per_sample = 16
        self.encoding = "PCM_S"
def _sf_info(p, *a, **k):
    return _SfInfo(p)
def _sf_load(p, frame_offset=0, num_frames=-1, *a, **k):
    d, sr = _sf.read(p, start=frame_offset,
                     frames=(-1 if num_frames in (-1, None) else num_frames),
                     dtype="float32", always_2d=True)
    return _torch.from_numpy(d.T.copy()), sr
_ta.info = _sf_info
_ta.load = _sf_load
'''
if "_sf_shim" not in s:
    s = s.replace("import openwakeword", SHIM + "\nimport openwakeword", 1)

tp.write_text(s)

# torchaudio.set_audio_backend больше не существует
for f in glob.glob("/usr/local/lib/python3*/dist-packages/torch_audiomentations/utils/io.py"):
    io = pathlib.Path(f)
    t = io.read_text()
    if "set_audio_backend" in t and "# patched" not in t:
        io.write_text(re.sub(r"^(\s*)(torchaudio\.set_audio_backend.*)$",
                             r"\1pass  # patched: \2", t, flags=re.M))
        print("io.py пропатчен:", f)

print("generate_samples.py:", pathlib.Path("/content/piper-sample-generator/generate_samples.py").exists())

правок generate_samples: 4 (ожидается 4)
io.py пропатчен: /usr/local/lib/python3.13/dist-packages/torch_audiomentations/utils/io.py
generate_samples.py: True


In [9]:
import importlib
mods = ["onnxruntime", "onnxscript", "piper", "torch", "torchaudio", "torchinfo",
        "torchmetrics", "speechbrain", "audiomentations", "torch_audiomentations",
        "acoustics", "pronouncing", "mutagen", "dp", "datasets", "tqdm",
        "scipy", "soundfile", "yaml"]
missing = []
for m in mods:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append((m, type(e).__name__))
print("не хватает:", missing or "ничего")
!nvidia-smi -L

не хватает: ничего
GPU 0: Tesla T4 (UUID: GPU-61a8a25b-48f7-4745-96bc-6f7f873be9f8)


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [21]:
!PYTHONPATH=/content/openwakeword:/content/piper-sample-generator python ./openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:CUDA available, using GPU
DEBUG:generate_samples:Batch 1/200 complete
DEBUG:generate_samples:Batch 2/200 complete
DEBUG:generate_samples:Batch 3/200 complete
DEBUG:generate_samples:Batch 4/200 complete
DEBUG:generate_samples:Batch 5/200 complete
DEBUG:generate_samples:Batch 6/200 complete
DEBUG:generate_samples:Batch 7/200 complete
DEBUG:generate_samples:Batch 8/200 complete
DEBUG:generate_samples:Batch 9/200 complete
DEBUG:generate_samples:Batch 10/200 complete
DEBUG:generate_samples:Batch 11/200 complete
DEBUG:generate_samples:Batch 12/200 complete
DEBUG:generate_samples:Batch 13/200 complete
DEBUG:generate_samples:Batch 14/200 complete
DEBUG:generate_samples:Batch 1

In [22]:
import glob, os, collections, numpy as np, soundfile as sf, scipy.signal as ss
bad = collections.Counter()
files = glob.glob("my_custom_model/**/*.wav", recursive=True)
for f in files:
    if sf.info(f).samplerate != 16000:
        bad[os.path.dirname(f)] += 1
        a, sr = sf.read(f)
        sf.write(f, ss.resample_poly(a, 16000, sr).astype(np.float32), 16000, subtype="PCM_16")
print(len(files), "файлов всего")
for d, n in bad.items():
    print("пересчитано", n, "в", d)

24000 файлов всего
пересчитано 2000 в my_custom_model/suzuya/positive_test
пересчитано 10000 в my_custom_model/suzuya/negative_train
пересчитано 2000 в my_custom_model/suzuya/negative_test
пересчитано 10000 в my_custom_model/suzuya/positive_train


In [23]:
SAVE = "/content/drive/MyDrive/juuzou-wake"
!tar -cf {SAVE}/clips.tar -C /content my_custom_model
!ls -lh {SAVE}

total 727M
-rw------- 1 root root 727M Sep  2 18:22 clips.tar
drwx------ 2 root root 4.0K Sep  2 17:36 v1-juuzou


In [24]:
!PYTHONPATH=/content/openwakeword:/content/piper-sample-generator python ./openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect a

In [25]:
!cp my_custom_model/suzuya/*features*.npy {SAVE}/
!ls -lh {SAVE}

total 868M
-rw------- 1 root root 727M Sep  2 18:22 clips.tar
-rw------- 1 root root  12M Sep  2 18:41 negative_features_test.npy
-rw------- 1 root root  59M Sep  2 18:41 negative_features_train.npy
-rw------- 1 root root  12M Sep  2 18:41 positive_features_test.npy
-rw------- 1 root root  59M Sep  2 18:41 positive_features_train.npy
drwx------ 2 root root 4.0K Sep  2 17:36 v1-juuzou


In [26]:
!PYTHONPATH=/content/openwakeword:/content/piper-sample-generator python ./openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

INFO:root:##################################################
Starting training sequence 1...
##################################################
Training: 100% 24999/25000 [11:06<00:00, 37.50it/s]
INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 2499/2500.0 [03:16<00:00, 12.74it/s]
INFO:root:##################################################
Starting training sequence 3...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 2499/2500.0 [03:14<00:00, 12.85it/s]
INFO:root:Merging checkpoints above the 90th percentile into single model...
INFO:root:
################
Final Model Accuracy: 0.7565000057220459
Final Model Recall: 0.5139999985694885
Final Model False Positives per Hour: 1.9469026327133179
##############

In [27]:
import glob, shutil, os
SAVE = "/content/drive/MyDrive/juuzou-wake"
found = sorted(glob.glob("/content/my_custom_model/**/*.onnx*", recursive=True))
print("нашлось:", found)
for f in found:
    shutil.copy(f, SAVE)
print("на Drive:", sorted(os.listdir(SAVE)))

нашлось: ['/content/my_custom_model/suzuya.onnx', '/content/my_custom_model/suzuya.onnx.data']
на Drive: ['clips.tar', 'negative_features_test.npy', 'negative_features_train.npy', 'positive_features_test.npy', 'positive_features_train.npy', 'suzuya.onnx', 'suzuya.onnx.data', 'v1-juuzou']


In [28]:
from google.colab import files
files.download("/content/drive/MyDrive/juuzou-wake/suzuya.onnx")
files.download("/content/drive/MyDrive/juuzou-wake/suzuya.onnx.data")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


In [23]:
import subprocess, glob, os
from IPython.display import Audio, display

CANDS = ["juuzou", "joozo", "juzo", "dzhuzo", "jooza", "juzzo", "dzuzo", "joo zo"]
os.makedirs("probe", exist_ok=True)
for c in CANDS:
    out = "probe/" + c.replace(" ", "_")
    !rm -rf {out} && mkdir -p {out}
    subprocess.run(["python", "/content/piper-sample-generator/generate_samples.py",
                    c, "--model", "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",
                    "--max-samples", "3", "--batch-size", "3", "--output-dir", out], check=False)
    for f in sorted(glob.glob(out + "/*.wav"))[:3]:
        print(c, f)
        display(Audio(f))

juuzou probe/juuzou/0.wav


juuzou probe/juuzou/1.wav


juuzou probe/juuzou/2.wav


joozo probe/joozo/0.wav


joozo probe/joozo/1.wav


...

dzuzo probe/dzuzo/1.wav


dzuzo probe/dzuzo/2.wav


joo zo probe/joo_zo/0.wav


joo zo probe/joo_zo/1.wav


joo zo probe/joo_zo/2.wav


In [9]:
!ls /content && ls /content/openwakeword /content/piper-sample-generator

drive  mit_rirs  openwakeword  piper-sample-generator  sample_data
/content/openwakeword:
benchmark     examples	 openwakeword		README.md
CHANGELOG.md  LICENSE	 openwakeword.egg-info	setup.py
docs	      notebooks  pyproject.toml		tests

/content/piper-sample-generator:
CHANGELOG.md  mypy.ini		      pylintrc	      script
LICENSE.md    piper_sample_generator  pyproject.toml  setup.cfg
models	      piper_train	      README.md


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!